# Wireless Coverage — Part 2b: Surfaces from a DSM Raster (the pixel-origin alternative)

**An alternative on-ramp.** Parts 1 and 2a start from the LiDAR *point cloud*. If instead you are
handed a **DSM raster** — a vendor product, a photogrammetric surface, a prior pipeline's output —
and no point cloud, this notebook gets you to the **same canonical surfaces**, so Part 3 onward is
identical either way.

With only a DSM there is no classified ground to triangulate, so we approximate the bare earth with
**pure raster ops**: a morphological *opening* of the DSM (`rst_filter` min then max — erosion then
dilation) suppresses above-ground structures and leaves an approximate terrain; `rst_chm` then gives
canopy/structure height (the "white top-hat" of the DSM).

> **This is an approximation, not a replacement for LiDAR ground.** A morphological opening cannot
> tell a flat-roofed building from a mesa, and it is applied per tile (mild seams at tile edges).
> **Part 2a's TIN bare earth, built from classified LiDAR ground returns, is the production DTM.**
> Use this path only when a point cloud is unavailable; for a production bare earth without LiDAR,
> use a bare-earth DEM instead — the one Part 1 derives from LiDAR, or a published terrain product (e.g. USGS 3DEP) — rather than this approximation.

> **Runtime.** Databricks Serverless environment 6 — the [lightweight tier](https://databrickslabs.github.io/geobrix/docs/api/execution-tiers) (pure Python/PySpark, no JAR or GDAL init script). Set `DEMO=False` to run the full San Francisco AOI.

---

**Last Update:** 2026-10-04

![Wireless Coverage series — config_nb → 01 → 02a/02b → 03a → 03b → 04](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/wireless-coverage/wireless-coverage-overview.png)

![Wireless Coverage Part 2b — DSM raster to approximate DTM + CHM via morphological opening](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/wireless-coverage/wireless-coverage-02b.png)

## Install GeoBrix

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [0]:
%restart_python

## Shared configuration

Shared imports, UDF registration, series parameters, and helpers are loaded from `config_nb` via `%run ./config_nb`. Notebook-specific overrides follow.

In [ ]:
%run ./config_nb

## Parameters

Shared series parameters come from `config_nb`. Part 2b reads DSM GeoTIFF tiles from `DSM_IN_DIR`
(default: the series' `OUT_DIR/dsm` — point it at your own DSM tiles if you have them) and writes the
canonical `wc_surface_{dsm,dtm,chm}` surfaces.

In [ ]:
# ── nb2b-specific overrides (after %run ./config_nb) ────────────────────────
# config_nb provides: F, rx, _tbl, _persist, plot_raster, lonlat_to_3857, glob,
#   X0, Y0, TILE_M, TPX, SRID, OUT_DIR, CATALOG, SCHEMA.
import os, re
TABLE_PREFIX    = "wc_surface"      # write: CATALOG.SCHEMA.wc_surface_{dsm,dtm,chm}[_demo]
DSM_IN_DIR      = f"{OUT_DIR}/dsm"  # DSM GeoTIFF tiles to start from (dsm_<tx>_<ty>.tif)
MORPH_KERNEL_PX = 25                # opening window (odd px); larger removes bigger structures
DEMO            = True              # True → Golden Gate Park ±2 tile window; False → full SF
FORCE_RELOAD    = False             # True → rebuild the surface tables (else reuse)
print(f"TABLE_PREFIX={TABLE_PREFIX} | DSM_IN_DIR={DSM_IN_DIR} | kernel={MORPH_KERNEL_PX}px | DEMO={DEMO}")

## Start from a DSM raster — read the DSM GeoTIFF tiles

A DSM-origin user has surface rasters, not a point cloud. We read the DSM GeoTIFF tiles from
`DSM_IN_DIR` and recover each tile's `(tx, ty)` grid index from its `dsm_<tx>_<ty>.tif` name.
(In the series these are the tiles Part 1 / Part 2a wrote; a real DSM-origin user points
`DSM_IN_DIR` at their own.)

In [ ]:
# List the DSM GeoTIFF tiles and recover (tx, ty) from each dsm_<tx>_<ty>.tif name.
_files = sorted(glob.glob(f"{DSM_IN_DIR}/dsm_*.tif"))
assert _files, (
    f"No DSM GeoTIFFs at {DSM_IN_DIR}. Point DSM_IN_DIR at your DSM tiles, "
    "or run Part 1 (or Part 2a) first to generate them."
)
_pat = re.compile(r"dsm_(-?\d+)_(-?\d+)\.tif$")
_rows = []
for _p in _files:
    _m = _pat.search(os.path.basename(_p))
    if _m:
        _rows.append((int(_m.group(1)), int(_m.group(2)), _p))

# Read each DSM tile (materialized so the surface table is self-contained for Parts 3-4).
dsm = (spark.createDataFrame(_rows, "tx int, ty int, path string")
            .select("tx", "ty", rx.rst_fromfile("path", materialize=True).alias("dsm")))

# DEMO: restrict to the Golden Gate Park ±2 tile window (matches the rest of the series).
if DEMO:
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)
    _dtx, _dty = int((_gx - X0) // TILE_M), int((_gy - Y0) // TILE_M)
    dsm = dsm.where(F.col("tx").between(_dtx - 2, _dtx + 2)
                    & F.col("ty").between(_dty - 2, _dty + 2))

# Rebuild the canonical surfaces only if absent (don't clobber a Part-2a run's tables/files).
_rebuild = FORCE_RELOAD or not spark.catalog.tableExists(_tbl(TABLE_PREFIX, "dtm"))
dsm = _persist(dsm, "dsm")   # carry the input DSM forward as the canonical wc_surface_dsm
print(f"{dsm.count()} DSM tiles from {DSM_IN_DIR} -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_dsm")

## Approximate bare earth — morphological opening of the DSM

A morphological **opening** (erosion then dilation over a square window) removes above-ground objects
smaller than the window while preserving the broader ground form. Two `rst_filter` passes — `min`
(erosion) then `max` (dilation) at `MORPH_KERNEL_PX` — approximate the bare earth. Larger kernels
remove larger structures but smooth real terrain and widen tile-edge seams; `rst_filter` works per
tile, so expect mild discontinuities at tile boundaries (a limitation the LiDAR path in Part 2a does
not have).

In [ ]:
# Opening = erosion (focal min) then dilation (focal max) over a square window. The result
# approximates bare earth; compute it once and persist as the canonical DTM.
K = MORPH_KERNEL_PX
dtm = dsm.select(
    "tx", "ty",
    rx.rst_filter(rx.rst_filter("dsm", F.lit(K), F.lit("min")), F.lit(K), F.lit("max")).alias("dtm"),
)
dtm = _persist(dtm, "dtm")   # -> wc_surface_dtm (morphological approximation of bare earth)
print(f"approx DTM {dtm.count()} tiles -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_dtm")

In [ ]:
# Preview the input DSM vs the approximated bare earth on a Golden Gate Park tile.
_gx, _gy = lonlat_to_3857(-122.486, 37.769)
_tx, _ty = int((_gx - X0) // TILE_M), int((_gy - Y0) // TILE_M)
_sel = (F.col("tx") == _tx) & (F.col("ty") == _ty)
dsm_t = dsm.where(_sel).first()
dtm_t = dtm.where(_sel).first()
print(f"sample tile tx={_tx} ty={_ty}")
if dsm_t: plot_raster(dsm_t["dsm"]["raster"], cmap="terrain", fig_w=7, fig_h=7)   # input DSM
if dtm_t: plot_raster(dtm_t["dtm"]["raster"], cmap="terrain", fig_w=7, fig_h=7)   # approx bare earth

## CHM — `rst_chm(DSM, approximated DTM)`

Difference the DSM and the approximated bare earth with `rst_chm` (DSM − DTM, clamped ≥ 0) — the
"white top-hat" of the DSM — and persist it to `CATALOG.SCHEMA.wc_surface_chm`.

In [ ]:
chm = dsm.join(dtm, ["tx", "ty"]).select("tx", "ty", rx.rst_chm("dsm", "dtm").alias("chm"))
chm = _persist(chm, "chm")   # -> wc_surface_chm
print(f"{chm.count()} CHM tiles -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_chm")
chm_t = chm.where(_sel).first()
if chm_t:
    plot_raster(chm_t["chm"]["raster"], cmap="viridis", fig_w=7, fig_h=7)

## Write the canonical surfaces to the Volume

Persist DSM, the approximated DTM, and CHM as path-backed GeoTIFF tiles under `OUT_DIR` (one
`gtiff_gbx` dir each) — the same canonical products Part 2a writes, so Part 3 (`03_h3_gridding`) and
Part 4 (`04_tower_viewsheds`) read them identically. We only (re)write when building fresh, so running
this alongside Part 2a does not overwrite that path's LiDAR-derived surfaces.

In [ ]:
if _rebuild:
    for _df, _kind in [(dsm, "dsm"), (dtm, "dtm"), (chm, "chm")]:
        (_df.select(F.concat_ws("_", F.lit(_kind), "tx", "ty").alias("source"),
                    F.col(_kind).alias("tile"))
            .write.format("gtiff_gbx").option("nameCol", "source").mode("overwrite").save(f"{OUT_DIR}/{_kind}"))
    print("wrote canonical surfaces ->", OUT_DIR, "(dsm, dtm, chm)")
else:
    print(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_dtm exists — kept (set FORCE_RELOAD=True to rebuild).")

## Summary

Starting from a **DSM raster** alone — no point cloud — we approximated the bare earth with a
morphological opening (`rst_filter` min→max) and derived canopy/structure height with `rst_chm` (the
white top-hat), landing the **same canonical `wc_surface_{dsm,dtm,chm}` surfaces** (tables + OUT_DIR
GeoTIFFs) that Part 2a produces from the point cloud. Part 3 and Part 4 are identical from here.

Remember this is a **pixel-only approximation**: the opening cannot tell a building roof from true
ground and is applied per tile. When a classified point cloud is available, **Part 2a's TIN bare
earth is the production DTM**; when it is not, pair the DSM with a published terrain product (e.g.
USGS 3DEP) for a production-grade bare earth.